# Learn to use Kedro

In [1]:
import pandas as pd

## 1. Set up your envrionment for Kedro

In order to use Kedro for your repository, you will need to set up your environment. To do so, we advise you to use `uv` and use the following command:
```sh
uvx kedro new --name <name_of_your_repo>
```
For this exercise, you won't need to create a Kedro repo as it is already prepared in this repo. However, you might need it for the personal project.
When creating your kedro you will have several tools to select or not. We recommend you to select the following tools each time you create a new repo:
1. Lint: Basic linting with ruff.
2. Test: Basic testing with pytest.
5. Data Folder: A folder structure for data management.

If you need, you can also add the other tools like 4) Docs: A Sphinx documentation setup or 6) PySpark: Configuration for working with PySpark, but it really depends on your needs.

```mermaid
Flowchart creating kedro repo

    A[Start]-->B[Enter Project Name]
    B-->C[Select Tools]
    C-->|None| D[None]
    C-->|Any combination| E[lint, test, logging, docs, data, pyspark]
    C-->|All| F[All]
    D-->G[Include Example Pipeline?]
    E-->G
    F-->G
    G-->|Yes| H[New Project Created?]
    G-->|No| H
```

Then, you can create a virtual environment and sync the dependencies. Select your environment and then sync:
```sh
uv sync
```

Finally, you can verify that your installation is correct:
```sh
uv run kedro info
```

## 2. Create your pipeline

The next step is to create your first pipeline in Kedro.
Let's take the example of a feature engineering pipeline.

In [6]:
df = pd.read_parquet('../data/fremotor1prem0304.parquet')
df.head()

,IDpol,Year,DrivAge,DrivGender,MaritalStatus,BonusMalus,LicenceNb,PayFreq,JobCode,VehAge,...,Garage,Area,Region,Channel,Marketing,PremTot,test_set,val_set,big_train_set,train_set
0,1000111.100,2003.0,44.0,F,Cohabiting,50.0,3.0,Half-yearly,Private employee,10.0,...,Closed zbox,A2,Headquarters,A,M1,144.1,1,0,0,0
1,1000113.100,2003.0,26.0,F,Cohabiting,85.0,2.0,Annual,Other,8.0,...,Opened collective parking,A7,Headquarters,A,M2,215.3,1,0,0,0
2,1000113.100,2003.0,27.0,F,Cohabiting,106.0,2.0,Half-yearly,Other,6.0,...,Opened collective parking,A7,Headquarters,A,M2,611.6,1,0,0,0
3,1000173.100,2003.0,52.0,M,Cohabiting,50.0,2.0,Half-yearly,Private employee,2.0,...,Closed zbox,A7,Headquarters,A,M1,415.2,0,0,1,1
4,1000173.101,2003.0,52.0,M,Cohabiting,50.0,2.0,Half-yearly,Private employee,1.0,...,Closed collective parking,A7,Headquarters,A,M3,487.8,0,0,1,1


First, create a feature engineering pipeline in this repository using the Kedro command:
```shell
kedro pipeline create feature_engineering
```
This command should create a `feature_engineering` folder in the `src/mlops_handbook/pipelines/` folder, that contains `__init__.py`, `nodes.py` and `pipeline.py` files. It also creates a `parameters_feature_engineering.yml` file that will contain the parameters of the functions, in a `conf/base/` folder. Finally, it also creates a `feature_engineering` folder in the `tests/pipelines` folder to contain the unit and integrations tests.

### Feature engineering

We have written several feature engineering lines to prepare the dataset before training a model.

In [ ]:
# Replace missing values with the mode (most frequent value)
df["MaritalStatus"] = df["MaritalStatus"].fillna(df["MaritalStatus"].mode()[0])
df["JobCode"] = df["JobCode"].fillna(df["JobCode"].mode()[0])

# Convert columns to the right type
df["Year"] = df["Year"].astype(int)
df["DrivAge"] = df["DrivAge"].astype(int)
df["DrivGender"] = df["DrivGender"].astype(str)
df["MaritalStatus"] = df["MaritalStatus"].astype(str)
df["BonusMalus"] = df["BonusMalus"].astype(int)
df["LicenceNb"] = df["LicenceNb"].astype(int)
df["JobCode"] = df["JobCode"].astype(str)
df["VehAge"] = df["VehAge"].astype(int)
df["VehGas"] = df["VehGas"].astype(str)
df["Area"] = df["Area"].astype(str)

The goal here is to create functions (nodes for Kedro) to contain the feature engineering pipeline.

**Exercise**: Create several nodes to do the following actions:
* Fill missing for specific columns using the most frequent value. The list of the columns to fill must be a parameter of the function.
* Convert columns to the right type. The function should accept, as parameters, the list of columns to convert to int, the list of columns to convert to str.

Now that you have created the functions and implemented them in the nodes file, you have to create the pipeline structure in the pipeline file.

**Exercise**: Orchestrate the functions in a Kedro pipeline:
* Create a Kedro pipeline to apply the different functions of the feature engineering part.
* Add the inputs and outputs of your pipeline in the `catalog.yml` file.
* Add the parameters of the pipeline in the `parameters_feature_engineering.yml` file.

<details>

<summary>Click to reveal tip for the inputs and outputs</summary>

In the catalog, you should add an element for the input `df_motor` for example with a path `data/01_raw/fremotor1prem0304.parquet`. You shoul also add another element `df_feature_engineered` for example with a path `data/04_feature/fremotor1prem0304.parquet`.

## Train model

Now, let's create kedro pipeline to train a model in order to predict the total premium.

**Exercise**: Create a kedro pipeline called `train_model`.

To train a model, we will use [scikit-learn](https://scikit-learn.org/stable/) pipelines in order to finish the feature engineering and train a model.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn import set_config

# Set transformer output to be a pandas DataFrame instead of numpy array
set_config(transform_output = "pandas")

In [ ]:
cat_columns = [
    "DrivGender", "MaritalStatus", # to complete
]
num_columns = [
    "Year", "DrivAge", # to complete
]

feature_engineering_pipeline = ColumnTransformer(
    transformers=[
        # (step name, transformer, column list to apply transformation to)
        ("onehot_encoding", LabelEncoder(), cat_columns),
        ("minmax_scaling", MinMaxScaler(), num_columns)
    ],
    remainder = "passthrough"
)

Let's train a Gradient Boosting model using the HistGradientBoosting using scikit-learn.

**Exercise**: Add a HistGradientBoosting model to the scikit-learn pipeline written before, in order to create a full training model scikit-learn pipeline.

Then, let's implement this function in Kedro.

**Exercise**: The goal is to implement the training pipeline in Kedro.
* 